# Урок 4.6: Агент управляет Metabase

В этой тетради:
1. Знакомство с Metabase REST API
2. Агент создаёт карточки (questions) в Metabase по запросу
3. Агент собирает дашборд из нескольких карточек

**Уровень Gen BI:** L5–L6 (автоматические дашборды)

**Metabase UI:** http://localhost:3000

## 1. Подключение к Metabase API

In [1]:
import os, json, requests
import pandas as pd
from openai import OpenAI

# Конфигурация Metabase. Все реквизиты — из env (compose пробрасывает
# MB_URL / MB_USER / MB_PASS в code-server). Никаких хардкод-паролей.
METABASE_URL = os.environ.get('MB_URL', 'http://metabase:3000')
# URL для печати ссылок в браузер (юзер открывает на хосте, не из docker-сети)
MB_PUBLIC_URL = os.environ.get('MB_PUBLIC_URL', 'http://localhost:3000')
MB_USER = os.environ['MB_USER']
MB_PASS = os.environ['MB_PASS']

# Аутентификация через session token
session = requests.post(f'{METABASE_URL}/api/session',
    json={'username': MB_USER, 'password': MB_PASS}).json()
MB_TOKEN = session.get('id', '')
MB_HEADERS = {'X-Metabase-Session': MB_TOKEN, 'Content-Type': 'application/json'}

print('Metabase session:', 'OK' if MB_TOKEN else 'FAILED')
print('API URL:', METABASE_URL, '| public URL:', MB_PUBLIC_URL, '| user:', MB_USER)

Metabase session: OK
API URL: http://metabase:3000 | public URL: http://localhost:3000 | user: admin@dddm.local


## 2. Metabase API: базовые операции

Разберём API вручную, чтобы понимать, что агент делает под капотом.

In [2]:
# Какие базы данных подключены?
dbs = requests.get(f'{METABASE_URL}/api/database', headers=MB_HEADERS).json()
for db in dbs.get('data', []):
    print(f"  ID={db['id']}: {db['name']} ({db['engine']})")

# Запомним ID нашей БД
DB_ID = dbs['data'][0]['id'] if dbs.get('data') else 1
print(f'\nИспользуем DB ID: {DB_ID}')

  ID=2: dddm_demo (postgres)
  ID=1: Sample Database (h2)

Используем DB ID: 2


In [3]:
# Создадим карточку (question) через API — чистый SQL
card = requests.post(f'{METABASE_URL}/api/card', headers=MB_HEADERS, json={
    'name': 'Выручка по категориям (API)',
    'dataset_query': {
        'type': 'native',
        'native': {
            'query': 'SELECT category AS "Категория", total_revenue AS "Выручка" FROM mart_category_stats ORDER BY total_revenue DESC'
        },
        'database': DB_ID
    },
    'display': 'bar',
    'visualization_settings': {
        'graph.dimensions': ['Категория'],
        'graph.metrics': ['Выручка']
    }
}).json()

print(f"Карточка создана: ID={card.get('id')}")
from IPython.display import HTML, display
_url = f"{MB_PUBLIC_URL}/question/{card.get('id')}"
display(HTML(f'🔗 <a href="{_url}" target="_blank" rel="noopener">{_url}</a>'))

Карточка создана: ID=80


**Откройте ссылку выше** — карточка с bar chart появилась в Metabase!

## 3. Агент, который создаёт карточки

Теперь автоматизируем: агент получает вопрос → генерирует SQL → создаёт карточку в Metabase.

In [4]:
tools = [
    {"type": "function", "function": {
        "name": "create_metabase_card",
        "description": "Создать карточку (вопрос + визуализация) в Metabase. Карточка сразу появляется в интерфейсе.",
        "parameters": {"type": "object", "properties": {
            "name": {"type": "string", "description": "Название карточки (русский)"},
            "sql_query": {"type": "string", "description": "SQL SELECT запрос"},
            "display": {"type": "string", "enum": ["table","bar","line","pie","area","scatter"], "description": "Тип визуализации"},
            "x_column": {"type": "string", "description": "Колонка для оси X / категорий"},
            "y_column": {"type": "string", "description": "Колонка для оси Y / значений"}
        }, "required": ["name","sql_query","display"]}
    }},
    {"type": "function", "function": {
        "name": "create_dashboard",
        "description": "Создать пустой дашборд в Metabase и добавить на него ранее созданные карточки.",
        "parameters": {"type": "object", "properties": {
            "name": {"type": "string", "description": "Название дашборда"},
            "card_ids": {"type": "array", "items": {"type": "integer"}, "description": "ID карточек для добавления"}
        }, "required": ["name","card_ids"]}
    }}
]

created_cards = []  # хранилище ID созданных карточек

def exec_create_card(name, sql_query, display, x_column=None, y_column=None):
    viz = {}
    if x_column: viz['graph.dimensions'] = [x_column]
    if y_column: viz['graph.metrics'] = [y_column]
    
    resp = requests.post(f'{METABASE_URL}/api/card', headers=MB_HEADERS, json={
        'name': name,
        'dataset_query': {'type': 'native', 'native': {'query': sql_query}, 'database': DB_ID},
        'display': display,
        'visualization_settings': viz
    })
    card = resp.json()
    card_id = card.get('id')
    created_cards.append(card_id)
    url = f'{MB_PUBLIC_URL}/question/{card_id}'
    try:
        from IPython.display import HTML, display
        display(HTML(f'🔗 <a href="{url}" target="_blank" rel="noopener">{url}</a>'))
    except Exception:
        pass
    print(f'  Карточка создана: {name} → {url}')
    return {'card_id': card_id, 'url': url}

def exec_create_dashboard(name, card_ids):
    dash = requests.post(f'{METABASE_URL}/api/dashboard', headers=MB_HEADERS,
                         json={'name': name}).json()
    dash_id = dash.get('id')
    # Добавляем карточки на дашборд
    cards_payload = []
    for i, cid in enumerate(card_ids):
        cards_payload.append({
            'id': -i-1, 'card_id': cid,
            'row': (i // 2) * 6, 'col': (i % 2) * 9,
            'size_x': 9, 'size_y': 6
        })
    requests.put(f'{METABASE_URL}/api/dashboard/{dash_id}',
                 headers=MB_HEADERS, json={'dashcards': cards_payload})
    url = f'{MB_PUBLIC_URL}/dashboard/{dash_id}'
    try:
        from IPython.display import HTML, display
        display(HTML(f'📊 <a href="{url}" target="_blank" rel="noopener">{url}</a>'))
    except Exception:
        pass
    print(f'  Дашборд создан: {name} → {url}')
    return {'dashboard_id': dash_id, 'url': url}

print('Metabase tools готовы')

Metabase tools готовы


In [5]:
# LLM-клиент (наш роутер) — креды из env, проброшены compose-ом
client = OpenAI(
    base_url=os.environ.get('AI_PROVIDER_URL', 'https://api.agentplatform.ru/v1'),
    api_key=os.environ['AI_PROVIDER_TOKEN'],
)
MODEL = os.environ.get('AI_PROVIDER_MODEL', 'anthropic/claude-opus-4.7')

SCHEMA = """Таблицы и витрины БД dddm_demo (Postgres). СТРОГО используй эти имена колонок — не выдумывай.

Витрины (готовые агрегаты, предпочтительны для дашбордов):
- mart_monthly_revenue(month, category, order_count, items_sold, revenue, cost, gross_profit)
- mart_category_stats(category, total_orders, total_items, total_revenue, avg_line_total, gross_profit, margin_pct)
- mart_channel_roi(channel, month, spend, clicks, cpc_actual, orders, revenue, roi_absolute, roi_pct)
- mart_customer_segments(city, segment, customers, orders, total_revenue, avg_order, revenue_per_customer)

Представления:
- v_funnel(week, page_views, searches, add_to_cart, checkouts, errors, conversion_pct)
- v_deal_pipeline(stage, manager, deal_count, total_amount, avg_deal, closed_count)
- v_order_details(order_id, order_date, customer_id, city, segment, channel, category, product, quantity, line_total)

Сырые таблицы (когда витрин не хватает):
- customers(customer_id, name, email, city, segment, registered_at, is_active)
- products(product_id, name, category, supplier, unit_cost, list_price)
- orders(order_id, customer_id, channel_id, order_date, status, total_amount)
- order_items(order_id, product_id, quantity, unit_price, discount_pct)
- channels(channel_id, name, cpc, description)   -- столбец называется name, НЕ channel_name
- marketing_spend(channel_id, month, spend, clicks)
- crm_deals(deal_id, customer_id, deal_name, stage, amount, created_at, closed_at, manager, source)
- system_logs(log_id, event_time, user_id, event_type, page_url, session_id, device, duration_ms, status_code)
"""

SYSTEM = f"""Ты — Gen BI агент с доступом к Metabase. Создавай карточки и дашборды по запросу.
Правила: SQL SELECT only, русские алиасы, ROUND числа, выбирай тип визуализации по смыслу.
Схема: {SCHEMA}
После создания — дай краткую интерпретацию."""

def ask_metabase(question):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(8):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice='auto')
        msg = resp.choices[0].message
        if not msg.tool_calls:
            if msg.content: print(msg.content)
            return
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if tc.function.name == 'create_metabase_card':
                result = exec_create_card(**args)
            elif tc.function.name == 'create_dashboard':
                result = exec_create_dashboard(**args)
            else:
                result = {'error': 'unknown'}
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

## 4. Агент создаёт карточку

In [6]:
ask_metabase("Создай карточку с трендом выручки по месяцам за 2024 год")

  Карточка создана: Тренд выручки по месяцам, 2024 → http://localhost:3000/question/81
Готово! Карточка создана: [открыть в Metabase](http://localhost:3000/question/81)

📈 **Тренд выручки по месяцам, 2024** — линейный график, агрегированный из витрины `mart_monthly_revenue` (выручка просуммирована по всем категориям внутри месяца). На графике сразу видна сезонность и динамика роста/спада по месяцам. Если нужно — могу разложить по категориям (stacked area) или добавить сравнение с прошлым годом.


**Откройте ссылку** — в Metabase появился line chart!

In [7]:
ask_metabase("Создай карточку с ROI маркетинговых каналов (bar chart)")

  Карточка создана: ROI маркетинговых каналов → http://localhost:3000/question/82
Готово! Карточка создана: [ROI маркетинговых каналов](http://localhost:3000/question/82)

**Что внутри:**
- Агрегация по каналам из витрины `mart_channel_roi` (суммы за все месяцы)
- Метрики: расходы, выручка, абсолютный ROI и ROI в %
- Сортировка по убыванию ROI %, чтобы сразу видеть лидеров и аутсайдеров

**Как читать:** столбцы показывают эффективность каждого канала — чем выше %, тем больше выручки приносит каждый вложенный рубль. Каналы с отрицательным ROI стоит рассмотреть на предмет оптимизации бюджета или отключения.


## 5. Агент собирает дашборд

In [8]:
# Агент создаёт несколько карточек и собирает дашборд
ask_metabase("""
Создай дашборд "Ежемесячный отчёт" с тремя карточками:
1. Тренд выручки по месяцам (line)
2. Выручка по категориям (bar)
3. Распределение заказов по каналам (pie)
""")

  Карточка создана: Тренд выручки по месяцам → http://localhost:3000/question/83


  Карточка создана: Выручка по категориям → http://localhost:3000/question/84


  Карточка создана: Распределение заказов по каналам → http://localhost:3000/question/85


  Дашборд создан: Ежемесячный отчёт → http://localhost:3000/dashboard/10
Готово! Дашборд **«Ежемесячный отчёт»** создан: http://localhost:3000/dashboard/10

**Состав:**
1. 📈 **Тренд выручки по месяцам** (line) — динамика выручки из `mart_monthly_revenue`, агрегированная по месяцам. Поможет увидеть сезонность и общий тренд роста/падения.
2. 📊 **Выручка по категориям** (bar) — сравнение категорий товаров из `mart_category_stats`, отсортировано по убыванию. Сразу видно категории-лидеры.
3. 🥧 **Распределение заказов по каналам** (pie) — доли каналов привлечения по числу заказов (join `orders` + `channels`). Покажет, какие каналы дают основной поток.

Если нужно — могу добавить KPI-карточки (общая выручка, средний чек, маржа) или фильтры по периоду/городу.


**Откройте дашборд** — 3 карточки на одной странице, обновляются автоматически!

## 6. Свой дашборд

In [ ]:
ask_metabase("Ваше задание: создайте дашборд с 4 карточками по теме CRM-воронка")

## Итоги

1. **Metabase API**: карточки и дашборды создаются программно через REST API
2. **Агент как интерфейс**: пользователь описывает на русском → агент генерирует SQL + визуализацию
3. **Регулярные дашборды**: созданные карточки обновляются автоматически при изменении данных
4. **Gen BI L5-L6**: от каскадных вычислений до полных дашбордов — всё через диалог